In [1]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
# Permite importar tu paquete de src/ sin instalarlo
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))

# 08b · Registro del modelo

> Continúa `07b_validacion_aprueba_o_no.ipynb` (**modelo final: regresión logística**, `C = 1`, umbral 1/6, sin calibrar; la prueba se abrió una sola vez y el modelo supera a «avisar a todos» con confianza).
> Dataset: `data/student_exam_performance.csv` (**aprueba con 70 o más; con menos de 70 pierde el curso**; clase de interés `Fail`). Plantilla: `notebooks/08_registro_modelo.ipynb` (esqueleto del curso, sin modificar). Ejemplo de referencia: `material-del-curso/ejemplo-creditsight/notebooks/08_registro_modelo.ipynb`.

En la etapa 6 hubo **muchas ejecuciones** (una por ensayo): eso es **tracking**. Ahora hay **un modelo final**, y lo que se guarda es otra cosa: una **versión del modelo**, con su firma, su ejemplo de entrada, sus métricas de la prueba y un **alias** (`champion`) que dice cuál es el vigente.

| | Tracking | Registro |
|---|---|---|
| Qué guarda | cada ensayo: parámetros y métricas | el modelo final, versionado |
| Cuántos hay | muchos (25 en la etapa 6) | pocos (uno por modelo que se decide usar) |
| Para qué | comparar y reproducir | saber **cuál** es el modelo en uso y poder volver a una versión anterior |

> **Seguridad:** un *pickle* **puede ejecutar código al cargarse**. Solo se cargan modelos que uno mismo (o el grupo) registró.

**Diferencias con el ejemplo CreditSight.** Allí el modelo tenía 3 clases y la unidad de partición era el cliente; aquí es **binario** (`Pass` / `Fail`), el pipeline recibe solo las **28 columnas** de la etapa 4 y cada fila es un estudiante distinto. Además se añade una sección que el ejemplo no tiene y el curso pide: **¿el modelo mejoró con las iteraciones?** (sección 6), calculada con los archivos de las etapas 4 a 7, sin escribir ninguna cifra a mano.

## 0. Conexión con el seguimiento

Dentro del servidor de Databricks no hay que configurar nada. Desde el computador se lee el `.env` (`PROFILE`, o `DATABRICKS_HOST` y `DATABRICKS_TOKEN`; ver `.env.example`). Si no hay conexión, o se agotó la cuota, se usa MLflow **local** (`mlflow_local.db`): el registro local cuenta igual y el notebook funciona igual. Esa es la mitad del plan B.

> **Si ves `local`:** no hay conexión con Databricks (sin `.env`, sin red o cuota agotada). **Si ves `databricks`:** el modelo queda en Unity Catalog (`catálogo.esquema.nombre`).

In [2]:
import json
from datetime import datetime

import joblib
import mlflow
import numpy as np
import pandas as pd
from mlflow import MlflowClient
from mlflow.models import infer_signature

from proyecto.data.cleaning import TARGET
from proyecto.features.engineering import STAGE4_DROP, feature_columns
from proyecto.tracking.mlflow_utils import configure_mlflow, registered_model_name, set_experiment

ROOT = Path.cwd().resolve().parent
REPORTS, MODELS, PROCESSED = ROOT / "reports", ROOT / "models", ROOT / "data" / "processed"

with warnings.catch_warnings(record=True) as avisos:
    warnings.simplefilter("always")
    donde = configure_mlflow()
for aviso in avisos:
    print("Aviso:", aviso.message)
experimento = set_experiment("proyecto-registro-aprueba-o-no")
NOMBRE = registered_model_name("estudiantes_aprueba_o_no")
print("MLflow:", donde)
print("Experimento:", experimento)
print("Nombre del modelo en el registro:", NOMBRE)

2026/11/05 20:25:36 INFO mlflow.tracking.fluent: Experiment with name 'proyecto-registro-aprueba-o-no' does not exist. Creating a new experiment.


MLflow: local
Experimento: proyecto-registro-aprueba-o-no
Nombre del modelo en el registro: estudiantes_aprueba_o_no


## 1. Cargar el modelo final y los resultados de la prueba

El pipeline final lo dejó el notebook 07b (`models/final_pipeline.joblib`, **no se versiona**: lo que viaja con el repo es el `model.joblib` que se exporta en el notebook 09 desde el registro). Las métricas salen de `reports/07_metricas_test.json` y las decisiones, de `reports/07_decisiones_congeladas.json`. Este notebook **no abre la prueba**: solo lee lo que la etapa 7 ya midió.

**Las columnas de entrada** son las 28 de la etapa 4 (`feature_columns(drop=STAGE4_DROP)`), todas conocidas **antes del examen**: nunca `exam_score`, `pass_status`, `performance_grade`, `performance_level`, `questions_correct`, `questions_attempted`, `student_id`, `gender` ni `family_income`.

In [3]:
modelo_final = joblib.load(MODELS / "final_pipeline.joblib")
resultados = json.loads((REPORTS / "07_metricas_test.json").read_text(encoding="utf-8"))
decisiones = json.loads((REPORTS / "07_decisiones_congeladas.json").read_text(encoding="utf-8"))
marca = (REPORTS / "prueba_abierta.txt").read_text(encoding="utf-8").strip()
assert resultados["decisiones"]["fecha_y_hora"] == decisiones["fecha_y_hora"], "las métricas y las decisiones congeladas no son de la misma ejecución"
print("modelo:", decisiones["modelo_final"], "· parámetros:", decisiones["parametros"], "· umbral:", round(decisiones["umbral"], 4), "· calibrado:", decisiones["calibrar"])
print("decisiones congeladas el:", decisiones["fecha_y_hora"])
print(marca)

train = joblib.load(PROCESSED / "train.joblib")                      # solo entrenamiento: sirve de ejemplo de entrada
numericas, categoricas = feature_columns(drop=STAGE4_DROP)
COLUMNAS_ENTRADA = numericas + categoricas
assert TARGET not in COLUMNAS_ENTRADA and len(COLUMNAS_ENTRADA) == 28
entrada = train[COLUMNAS_ENTRADA].head(200).copy()
entrada[numericas] = entrada[numericas].astype("float64")           # un entero con vacíos no cabe en la firma: todo numérico a float
print("firma sobre", len(entrada), "filas de ejemplo,", entrada.shape[1], "columnas de entrada:", len(numericas), "numéricas y", len(categoricas), "categóricas")

modelo: regresión logística · parámetros: {'C': 1.0} · umbral: 0.1667 · calibrado: False
decisiones congeladas el: 2026-11-05T20:12:56
Prueba abierta el 2026-11-05T20:12:58. Modelo regresión logística, calibrar=False, umbral 0.1667. Una sola vez.


firma sobre 200 filas de ejemplo, 28 columnas de entrada: 12 numéricas y 16 categóricas


## 2. Qué se registra

Junto al modelo viajan:

- los **parámetros** y las **decisiones** (qué modelo, `C`, umbral, si se calibró, con qué datos se decidió);
- las **métricas de la prueba**, las mismas de la tabla del notebook 07b (con prefijo `test_`);
- la **firma** (columnas y tipos de entrada, probabilidades de salida) y un **ejemplo de entrada**;
- **etiquetas** para encontrarlo después (cuándo se congelaron las decisiones, si supera a «avisar a todos»).

> **Nota técnica:** hay que pasar `serialization_format="cloudpickle"`. El formato por defecto de MLflow rechaza guardar un `Pipeline` con transformadores propios (error *untrusted types*).

In [4]:
por_medida = {fila["medida"]: fila for fila in resultados["intervalos_95"]}
diferencia = por_medida["costo: modelo − «avisar a todos»"]
metricas_prueba = {f"test_{k}": float(v) for k, v in resultados["prueba"].items()}
metricas_prueba.update({
    "test_cost_ic95_inf": float(por_medida["costo por estudiante (menor es mejor)"]["IC 95 % inferior"]),
    "test_cost_ic95_sup": float(por_medida["costo por estudiante (menor es mejor)"]["IC 95 % superior"]),
    "test_cost_avisar_a_todos": float(resultados["baseline_avisar_a_todos"]["cost"]),
    "test_cost_diff_vs_avisar_a_todos": float(diferencia["valor"]),
    "test_cost_diff_ic95_inf": float(diferencia["IC 95 % inferior"]),
    "test_cost_diff_ic95_sup": float(diferencia["IC 95 % superior"]),
})
metricas_prueba = {k: v for k, v in metricas_prueba.items() if np.isfinite(v)}

firma = infer_signature(entrada, modelo_final.predict_proba(entrada))
with mlflow.start_run(run_name="modelo-final") as corrida:
    mlflow.log_params({"modelo": decisiones["modelo_final"], "C": decisiones["parametros"]["C"], "umbral": round(decisiones["umbral"], 6),
                       "calibrado": decisiones["calibrar"], "n_variables": len(COLUMNAS_ENTRADA)})
    mlflow.log_metrics(metricas_prueba)
    mlflow.set_tags({"etapa": "08_registro", "decisiones_congeladas": decisiones["fecha_y_hora"], "unidad_de_particion": "fila (student_id único)",
                     "supera_baseline": str(resultados["supera_al_baseline_con_confianza"]["ambas"]),
                     "nota": "prueba abierta una sola vez en la etapa 7; costos 5 : 1 son un supuesto"})
    mlflow.sklearn.log_model(
        modelo_final, name="model", signature=firma, input_example=entrada.head(5),
        serialization_format="cloudpickle",   # ver la nota técnica: sin esto MLflow rechaza un Pipeline completo
        registered_model_name=NOMBRE,
    )
    id_corrida = corrida.info.run_id

cliente = MlflowClient()
version = max(int(v.version) for v in cliente.search_model_versions(f"name = '{NOMBRE}'"))
print("ejecución:", id_corrida)
print(f"modelo registrado: {NOMBRE} · versión {version} · {len(metricas_prueba)} métricas de la prueba")

2026/11/05 20:25:37 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


2026/11/05 20:25:40 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


ejecución: 29f92c8497174db78e90cd12220ca183
modelo registrado: estudiantes_aprueba_o_no · versión 1 · 32 métricas de la prueba


Successfully registered model 'estudiantes_aprueba_o_no'.
Created version '1' of model 'estudiantes_aprueba_o_no'.


## 3. Poner el alias `champion`

Una **versión** es un número que no dice nada sobre su uso. El **alias** sí: `champion` es «el modelo vigente». Quien consuma el modelo pide `models:/<nombre>@champion` y no necesita saber qué número de versión es. Si mañana hay una versión mejor, se mueve el alias y nadie cambia su código; si algo sale mal, se mueve de vuelta.

In [5]:
cliente.set_registered_model_alias(NOMBRE, "champion", str(version))
cliente.set_model_version_tag(NOMBRE, str(version), "decisiones_congeladas", decisiones["fecha_y_hora"])
champion = cliente.get_model_version_by_alias(NOMBRE, "champion")
print(f"alias champion → versión {champion.version}  (ejecución {champion.run_id[:8]}…)")

alias champion → versión 1  (ejecución 29f92c84…)


## 4. Comprobación de ida y vuelta

Se **carga desde el registro** (por alias) y se comprueba que predice **exactamente igual** que el objeto que se registró, con 500 estudiantes del entrenamiento. Si no, lo que está en el registro no es lo que se evaluó en la etapa 7. También se comprueba que lo recuperado conserva el mismo **umbral de decisión** al aplicarlo (las mismas alertas).

In [6]:
desde_registro = mlflow.sklearn.load_model(f"models:/{NOMBRE}@champion")
muestra = train[COLUMNAS_ENTRADA].head(500).copy()
muestra[numericas] = muestra[numericas].astype("float64")
esperado, obtenido = modelo_final.predict_proba(muestra)[:, 1], desde_registro.predict_proba(muestra)[:, 1]
ida_y_vuelta = bool(np.allclose(esperado, obtenido))
mismas_alertas = bool(((esperado >= decisiones["umbral"]) == (obtenido >= decisiones["umbral"])).all())
diferencia_maxima = float(np.abs(esperado - obtenido).max())
print(f"las 500 predicciones coinciden: {ida_y_vuelta} (diferencia máxima {diferencia_maxima:.2e}) · mismas alertas con el umbral {decisiones['umbral']:.3f}: {mismas_alertas}")
assert ida_y_vuelta and mismas_alertas, "El modelo del registro no coincide con el que se evaluó"

las 500 predicciones coinciden: True (diferencia máxima 0.00e+00) · mismas alertas con el umbral 0.167: True


## 5. Guardar la constancia

Queda en `reports/08_registro.json` (y se resume en `reports/08_registro.md`): dónde se registró, qué versión, qué alias y con qué decisiones. La app **no consulta el registro**; trabaja con el `model.joblib` que se exporta en el notebook 09.

In [7]:
constancia = {
    "fecha_y_hora": datetime.now().isoformat(timespec="seconds"), "donde": donde, "experimento": experimento,
    "modelo_registrado": NOMBRE, "version": version, "alias": "champion", "ejecucion": id_corrida,
    "ida_y_vuelta_ok": ida_y_vuelta, "diferencia_maxima": diferencia_maxima,
    "modelo": decisiones["modelo_final"], "parametros": decisiones["parametros"], "umbral": decisiones["umbral"], "calibrado": decisiones["calibrar"],
    "columnas_entrada": COLUMNAS_ENTRADA, "decisiones_congeladas": decisiones["fecha_y_hora"],
}
(REPORTS / "08_registro.json").write_text(json.dumps(constancia, indent=2, ensure_ascii=False), encoding="utf-8")
print({k: v for k, v in constancia.items() if k != "columnas_entrada"})

{'fecha_y_hora': '2026-11-05T20:25:40', 'donde': 'local', 'experimento': 'proyecto-registro-aprueba-o-no', 'modelo_registrado': 'estudiantes_aprueba_o_no', 'version': 1, 'alias': 'champion', 'ejecucion': '29f92c8497174db78e90cd12220ca183', 'ida_y_vuelta_ok': True, 'diferencia_maxima': 0.0, 'modelo': 'regresión logística', 'parametros': {'C': 1.0}, 'umbral': 0.16666666666666666, 'calibrado': False, 'decisiones_congeladas': '2026-11-05T20:12:56'}


## 6. ¿El modelo mejoró con las iteraciones?

Cada etapa cambió algo: variables (4), modelo (5), parámetros y umbral (6), calibración (7). La pregunta honesta es **cuáles de esos cambios mejoraron el costo de forma distinguible del ruido** y cuáles no. Se arma la tabla **leyendo los archivos de cada etapa** (nada escrito a mano).

**Cómo se lee:** el costo es por estudiante (menor es mejor). Las filas **no son comparables entre sí cuando usan estudiantes distintos** (validación cruzada con 80.000, reservados con 20.000, prueba con otros 20.000): ±0,003 entre subconjuntos es normal. Por eso cada fila se compara **contra la anterior en los mismos estudiantes**, con la regla de cada etapa: en validación cruzada, mejora / ruido entre pliegues (≥ 1 = mejora; entre −1 y 1 = no se distingue del ruido); con reservados o prueba, el intervalo del 95 % de la diferencia.

In [8]:
t04 = pd.read_csv(REPORTS / "04_features_tabla.csv", index_col=0)
t05 = pd.read_csv(REPORTS / "05_torneo_tabla.csv", index_col=0)
p06 = json.loads((REPORTS / "06_best_params.json").read_text(encoding="utf-8"))
ruido = lambda mejora, r: "mejora" if mejora / r >= 1 else ("empeora" if mejora / r <= -1 else "no se distingue del ruido")
por_intervalo = lambda bajo, alto: "mejora" if alto < 0 else ("empeora" if bajo > 0 else "no se distingue del ruido")

base = t05.loc["línea base (avisar a todos)"]
ref = t04.loc["referencia (35 variables)"]
final4 = t04.loc["8 propuesta final (ideas 3 + 7 juntas)"]
logis = t05.loc["regresión logística"]
cal = decisiones["calibracion"]
d_opt, (b_opt, a_opt) = p06["diferencia_costo"]["valor"], p06["diferencia_costo"]["intervalo_95"]
res_prueba = resultados["prueba"]["cost"]
ic_prueba = por_medida["costo por estudiante (menor es mejor)"]
dif_prueba = por_medida["costo: modelo − «avisar a todos»"]

filas = [
    ("0 · Sin modelo: «avisar a todos»", "—", "validación cruzada, 80.000", base["costo"], f"± {base['± pliegues']:.4f}", "punto de partida"),
    ("4 · Modelo con las 35 variables (LightGBM)", "primer modelo real", "validación cruzada, 80.000", ref["costo"], f"± {ref['± ruido']:.4f}",
     ruido(base["costo"] - ref["costo"], ref["± ruido"]) + f" frente a «avisar a todos» (−{base['costo'] - ref['costo']:.3f})"),
    ("4 · Propuesta final: 28 variables (LightGBM)", "−7 variables sin señal, sin indicador de vacío", "validación cruzada, 80.000", final4["costo"], f"± {final4['± ruido']:.4f}",
     ruido(ref["costo"] - final4["costo"], final4["± ruido"]) + " (pero el modelo es más simple)"),
    ("5 · Torneo: regresión logística (28 variables)", "cambio de modelo", "validación cruzada, 80.000", logis["costo"], f"± {logis['± pliegues']:.4f}",
     ruido(final4["costo"] - logis["costo"], logis["± pliegues"]) + " (empate con LightGBM; gana por simplicidad)"),
    ("6 · Optimización: partida (C = 1, umbral 1/6)", "—", "20.000 reservados", p06["costo_reservados"]["partida"], "—", "punto de partida de la comparación"),
    ("6 · Optimización: C y umbral optimizados", "25 ensayos de Optuna", "20.000 reservados", p06["costo_reservados"]["optimizado"], f"IC dif. [{b_opt:+.4f}; {a_opt:+.4f}]",
     por_intervalo(b_opt, a_opt)),
    ("7 · Calibración (sigmoide)", "probabilidades recalibradas", "20.000 reservados", cal["costo_con"], f"IC dif. [{cal['intervalo_95'][0]:+.4f}; {cal['intervalo_95'][1]:+.4f}]",
     por_intervalo(*cal["intervalo_95"]) + " (no se calibró)"),
    ("7 · PRUEBA, abierta una sola vez", "modelo final, sin cambios", "20.000 de la prueba", res_prueba, f"IC [{ic_prueba['IC 95 % inferior']:.4f}; {ic_prueba['IC 95 % superior']:.4f}]",
     por_intervalo(dif_prueba["IC 95 % inferior"], dif_prueba["IC 95 % superior"]) + f" frente a «avisar a todos» ({dif_prueba['valor']:+.4f})"),
]
evolucion = pd.DataFrame(filas, columns=["etapa", "qué se cambió", "con qué estudiantes", "costo", "incertidumbre", "¿mejoró? (frente a lo anterior)"]).set_index("etapa")
display(evolucion.style.format({"costo": "{:.4f}"}))
evolucion.to_csv(REPORTS / "08_evolucion_modelo.csv", encoding="utf-8")

salto = base["costo"] - res_prueba
print(f"Salto real: de «sin modelo» ({base['costo']:.4f}) a modelo en la prueba ({res_prueba:.4f}): −{salto:.4f} por estudiante ({salto / base['costo']:.1%}).")
print(f"Mejora lograda por las iteraciones posteriores al primer modelo (variables, torneo, optimización, calibración): {ref['costo'] - res_prueba:+.4f} entre subconjuntos distintos (ruido típico ± 0,003) y ninguna distinguible del ruido en los mismos estudiantes.")
print(f"Prueba frente a reservados: {res_prueba:.4f} frente a {p06['costo_reservados']['partida']:.4f} (diferencia {res_prueba - p06['costo_reservados']['partida']:+.4f}): elegir con validación no infló el resultado.")

,qué se cambió,con qué estudiantes,costo,incertidumbre,¿mejoró? (frente a lo anterior)
etapa,,,,,
0 · Sin modelo: «avisar a todos»,—,"validación cruzada, 80.000",0.3079,± 0.0000,punto de partida
4 · Modelo con las 35 variables (LightGBM),primer modelo real,"validación cruzada, 80.000",0.2579,± 0.0048,mejora frente a «avisar a todos» (−0.050)
4 · Propuesta final: 28 variables (LightGBM),"−7 variables sin señal, sin indicador de vacío","validación cruzada, 80.000",0.2584,± 0.0030,no se distingue del ruido (pero el modelo es más simple)
5 · Torneo: regresión logística (28 variables),cambio de modelo,"validación cruzada, 80.000",0.2576,± 0.0032,no se distingue del ruido (empate con LightGBM; gana por simplicidad)
"6 · Optimización: partida (C = 1, umbral 1/6)",—,20.000 reservados,0.2547,—,punto de partida de la comparación
6 · Optimización: C y umbral optimizados,25 ensayos de Optuna,20.000 reservados,0.2549,IC dif. [-0.0014; +0.0014],no se distingue del ruido
7 · Calibración (sigmoide),probabilidades recalibradas,20.000 reservados,0.2547,IC dif. [-0.0008; +0.0006],no se distingue del ruido (no se calibró)
"7 · PRUEBA, abierta una sola vez","modelo final, sin cambios",20.000 de la prueba,0.2550,IC [0.2467; 0.2634],mejora frente a «avisar a todos» (-0.0529)


Salto real: de «sin modelo» (0.3079) a modelo en la prueba (0.2550): −0.0529 por estudiante (17.2%).
Mejora lograda por las iteraciones posteriores al primer modelo (variables, torneo, optimización, calibración): +0.0029 entre subconjuntos distintos (ruido típico ± 0,003) y ninguna distinguible del ruido en los mismos estudiantes.
Prueba frente a reservados: 0.2550 frente a 0.2547 (diferencia +0.0003): elegir con validación no infló el resultado.


### Lo que muestran los números

- **El modelo sí mejoró frente a no tener modelo, y es la única mejora real.** El primer modelo ya redujo el costo de 0,308 a ≈ 0,258 (diez veces el ruido). La prueba final lo confirma con estudiantes nuevos: 0,2550, ahorro de ≈ 17 % con el intervalo de la diferencia por debajo de cero.
- **Las iteraciones siguientes no mejoraron el costo de forma medible:** quitar 7 variables sin señal (−0,0005), cambiar de LightGBM a regresión logística (+0,0008), optimizar `C` y umbral (+0,0001, intervalo que cruza el cero) y calibrar (0,0000) quedaron todas dentro del ruido. **No fueron tiempo perdido:** el modelo final es más simple (28 variables y un modelo lineal que se explica con coeficientes), está calibrado y se comprobó que no había nada más que ganar con `C`, el umbral o la calibración.
- **La única variable que movió el costo fue `exam_difficulty`** (sin ella el costo sube de 0,258 a 0,283): decide el *valor* del modelo y solo es válida si se conoce antes de avisar. Es un supuesto de negocio pendiente.
- **El registro (esta etapa) no cambió el modelo:** la ida y vuelta da la misma predicción (diferencia máxima ≈ 0), así que lo que se registró es exactamente lo que se evaluó en la prueba.
- **Qué no cambió:** el modelo sigue avisando a ≈ 90 de cada 100 estudiantes y la brecha de alertas por ingreso (93,2 % en `Low` frente a 84,0 % en `High`) es la de la etapa 7. Las iteraciones no la corrigieron porque no se buscó corregirla.

## Entregable de esta etapa

- [ ] El modelo está **registrado** con parámetros, métricas de la prueba, firma, ejemplo de entrada y etiquetas.
- [ ] El alias **`champion`** apunta a la versión registrada.
- [ ] La **ida y vuelta** pasa: el modelo recuperado del registro predice igual que el evaluado.
- [ ] `reports/08_registro.json` y `reports/08_registro.md` están completos, y la tabla de `reports/08_evolucion_modelo.csv` responde si el modelo mejoró con las iteraciones.
- [ ] Se dice con claridad **dónde** quedó registrado (Databricks o local) y por qué.

## Qué sigue

**Notebook 09:** exportar `models/model.joblib` desde `champion`, escribir el esquema de entrada y comprobar que la función de predicción rechaza los datos imposibles. Con eso la app ya puede cargar el modelo. **La app no consulta Databricks**: no se ponen credenciales en la nube, y el despliegue no depende de la cuota de la plataforma. El registro sigue siendo la **fuente de verdad**: el archivo siempre se exporta desde la versión `champion`.